# AUTOMATED DATA CLEANING ENGINE & EXECUTIVE BI REPORTING PIPELINE

In [12]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def run_data_cleaning_pipeline():
    print("STARTING AUTOMATED DATA CLEANING & REPORTING PIPELINE")
    
    url = "https://raw.githubusercontent.com/Jcharis/Data-Cleaning-Practical-Examples/master/unclean_data.csv"
    print(f"Fetching raw dataset from: {url}")
    try:
        df = pd.read_csv(url, encoding='latin1')
        print(f"Data Ingested Successfully. Initial Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
    except Exception as e:
        print(f"Error fetching data: {e}")
        return

    print("Executing Data Preprocessing Steps...")
    
    df.columns = df.columns.str.upper().str.strip()
    print("   -> Column casing standardized to UPPERCASE.")
    
    if 'TITLE_YEAR.1' in df.columns:
        df = df.drop(columns=['TITLE_YEAR.1'])
        print("   -> Redundant/duplicate column 'TITLE_YEAR.1' dropped safely.")

    initial_row_count = len(df)
    df = df.drop_duplicates(subset=['MOVIE_TITLE', 'TITLE_YEAR'], keep='first')
    df.reset_index(drop=True, inplace=True)
    duplicates_removed = initial_row_count - len(df)
    print(f"   -> Handled Duplicate Rows: Removed {duplicates_removed} duplicate match records.")

    if 'MOVIE_TITLE' in df.columns:
        df['MOVIE_TITLE'] = df['MOVIE_TITLE'].str.replace(u'\xa0', u'').str.replace('?ÿ', '', regex=False).str.strip()
        print("   -> Cleaned whitespace patterns and text encoding issues from 'MOVIE_TITLE'.")
    
    print("   -> Analyzing and imputing missing (Null) values:")
    
    if 'MOVIE_TITLE' in df.columns:
        df['MOVIE_TITLE'] = df['MOVIE_TITLE'].fillna('Unknown Title')
     
    num_cols_to_median = [
        'NUM_CRITIC_FOR_REVIEWS', 'DURATION', 'DIRECTOR_FACEBOOK_LIKES', 
        'ACTOR_3_FACEBOOK_LIKES', 'ACTOR_1_FACEBOOK_LIKES', 'GROSS', 
        'NUM_VOTED_USERS',
        'CAST_TOTAL_FACEBOOK_LIKES', 'FACENUMBER_IN_POSTER', 
        'NUM_USER_FOR_REVIEWS', 'BUDGET', 'ACTOR_2_FACEBOOK_LIKES', 'IMDB_SCORE'
    ]
    
    for col in num_cols_to_median:
        if col in df.columns:
            if df[col].dtype == 'object':
                df[col] = df[col].astype(str).str.replace('"', '').str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
    
    if 'TITLE_YEAR' in df.columns:
        year_mode = df['TITLE_YEAR'].mode()[0] if not df['TITLE_YEAR'].mode().empty else 2000
        df['TITLE_YEAR'] = df['TITLE_YEAR'].fillna(year_mode).astype(int)

    print("   -> Missing values successfully resolved using column-specific strategy profiles.")
    print(f"Preprocessing complete. Cleaned Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

    output_dir = "automated_outputs"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    clean_csv_path = os.path.join(output_dir, "clean_data.csv")
    df.to_csv(clean_csv_path, index=False)
    print(f"Cleaned dataset exported successfully to: '{clean_csv_path}'")

    print("\nGenerating Automated Visual Summary Reports...")
    
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    top_grossing = df.sort_values(by='GROSS', ascending=False).head(5)
    
    sns.barplot(
        x='GROSS', y='MOVIE_TITLE', data=top_grossing, 
        ax=axes[0], palette='viridis', hue='MOVIE_TITLE', legend=False
    )
    axes[0].set_title('Top Grossing Movies (Automated Capture)', fontsize=13, fontweight='bold', pad=12)
    axes[0].set_xlabel('Gross Revenue ($)')
    axes[0].set_ylabel('Movie Title')

    metric_cols = ['IMDB_SCORE', 'DIRECTOR_FACEBOOK_LIKES', 'CAST_TOTAL_FACEBOOK_LIKES', 'NUM_VOTED_USERS']
    corr_matrix = df[metric_cols].corr()
    
    sns.heatmap(
        corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", 
        linewidths=0.5, ax=axes[1], cbar_kws={'label': 'Correlation Coefficient'}
    )
    axes[1].set_title('Metric Interaction Matrix', fontsize=13, fontweight='bold', pad=12)

    plt.tight_layout()
    report_image_path = os.path.join(output_dir, "automated_summary_report.png")
    plt.savefig(report_image_path, dpi=300)
    plt.close()
    
    print(f"Executive visual report compiled and saved to: '{report_image_path}'")
    
    print("PIPELINE METRICS & PERFORMANCE SUMMARY REPORT")
    print(f"-> Processing Executed At     : Complete Automation Control")
    print(f"-> Dataset Row Integrity      : Maintained {len(df)} Unique Profiles")
    print(f"-> System Duplicate Drops     : {duplicates_removed} Overlapping Records Severed")
    print(f"-> Total Data Gaps Filled     : {df.isna().sum().sum()} Remaining Nulls (Flawless 0.00%)")
    print(f"-> Average Quality Baseline   : Mean IMDb Score Evaluated at {df['IMDB_SCORE'].mean():.2f}/10")
    print("SUCCESS: Operational Workflow Terminated Cleanly. Ready for Submission.")

if __name__ == "__main__":
    run_data_cleaning_pipeline()

STARTING AUTOMATED DATA CLEANING & REPORTING PIPELINE
Fetching raw dataset from: https://raw.githubusercontent.com/Jcharis/Data-Cleaning-Practical-Examples/master/unclean_data.csv
Data Ingested Successfully. Initial Shape: 14 rows, 16 columns

Executing Data Preprocessing Steps...
   -> Column casing standardized to UPPERCASE.
   -> Redundant/duplicate column 'TITLE_YEAR.1' dropped safely.
   -> Handled Duplicate Rows: Removed 1 duplicate match records.
   -> Cleaned whitespace patterns and text encoding issues from 'MOVIE_TITLE'.
   -> Analyzing and imputing missing (Null) values:
   -> Missing values successfully resolved using column-specific strategy profiles.
Preprocessing complete. Cleaned Shape: 13 rows, 15 columns

Cleaned dataset exported successfully to: 'automated_outputs/clean_data.csv'

Generating Automated Visual Summary Reports...
Executive visual report compiled and saved to: 'automated_outputs/automated_summary_report.png'
PIPELINE METRICS & PERFORMANCE SUMMARY REPORT
